# Using napari-TurboBox from Python

This tutorial shows the Python API of napari-TurboBox on a small synthetic volume:

1. a box layer in one viewer: adding, validating, editing, undo;
2. what a 2D view draws (only the boxes on its slice);
3. the synchronized four-viewer layout (XY editable; XZ, YZ and 3D read-only) and the shared box store;
4. listening to changes and bulk edits;
5. export to COCO, YOLO and the native format, and import;
6. using boxes as prompts (cropping, axis order);
7. 2D images and the control panel.

**Requirements:** napari >= 0.6.6, < 0.10 with a Qt binding and napari-turbobox (`pip install -e ..` from this folder).
Viewers are created hidden (`SHOW = False`), so the notebook also runs unattended; set `SHOW = True` to get the napari windows.

In [1]:
SHOW = False  # True: open the napari windows (needs a desktop session)

if SHOW:
    get_ipython().run_line_magic("gui", "qt")

import json
import tempfile
from pathlib import Path

import napari
import numpy as np

import napari_turbobox
from napari_turbobox import BBoxDataStore, TurboBoxLayer, create_synchronized_bbox_layers
from napari_turbobox.export import boxes_to_coco, boxes_to_yolo

print("napari", napari.__version__)

napari 0.7.0


## A synthetic volume

Twelve blurred blobs in a `(z, y, x) = (40, 128, 128)` volume, and one box around each blob.

In [2]:
rng = np.random.default_rng(0)
shape = (40, 128, 128)  # (z, y, x)
zz, yy, xx = np.ogrid[: shape[0], : shape[1], : shape[2]]
centres = rng.uniform([5, 15, 15], [35, 113, 113], size=(12, 3))
volume = np.zeros(shape, np.float32)
for cz, cy, cx in centres:
    volume += np.exp(-(((zz - cz) / 3) ** 2 + ((yy - cy) / 6) ** 2 + ((xx - cx) / 6) ** 2))

half = np.array([4, 8, 8])
boxes = np.stack([np.floor(centres - half), np.ceil(centres + half)], axis=1)
boxes = np.clip(boxes, 0, np.array(shape) - 1)
boxes.shape  # (N, 2, D): [[z_min, y_min, x_min], [z_max, y_max, x_max]] per box

(12, 2, 3)

## 1. A box layer in one viewer

`TurboBoxLayer` is a napari Shapes layer that displays boxes stored as an `(N, 2, D)` array.
Boxes are **inclusive voxel index ranges** in napari axis order: `min` and `max` are the first and last voxel index on every axis.
Before validation, `add_boxes` and the `bounding_boxes` setter expand a flat hidden axis: with `image_shape` set, that axis spans the full image depth (`0` to `shape - 1`). Here z is hidden, so a box with zero extent in z becomes a box through the whole volume.
Validation then checks image bounds and the minimum extent: `0 <= min`, `max <= shape - 1` and `max - min >= 1` on every axis. Zero extent on a displayed axis is rejected.

In [3]:
viewer = napari.Viewer(show=SHOW)
viewer.add_image(volume, name="volume")
layer = TurboBoxLayer(ndim=3, image_shape=volume.shape, name="boxes", edge_color="yellow")
viewer.add_layer(layer)

layer.add_boxes(boxes)
print(len(layer.bounding_boxes), "boxes,", layer.bounding_boxes.dtype)
layer.bounding_boxes[:2]  # a copy; the layer's store is not changed through it

12 boxes, float32


array([[[ 20.,  33.,  11.],
        [ 29.,  50.,  28.]],

       [[  1.,  86.,  96.],
        [ 10., 103., 113.]]], dtype=float32)

In [4]:
# Out-of-bounds coordinates and zero extent on a displayed axis are rejected.
for bad in (
    [[[0, 0, 0], [10, 10, 200]]],  # x_max exceeds shape - 1
    [[[5, 5, 5], [20, 5, 20]]],    # zero extent along displayed y
):
    try:
        layer.add_boxes(bad)
    except ValueError as err:
        print("rejected:", err)
    else:
        raise AssertionError("invalid box accepted")
assert len(layer.bounding_boxes) == len(boxes)
print(len(layer.bounding_boxes), "boxes")

rejected: Box 0 axis 2: max (200.0) exceeds image_shape - 1 (127)
rejected: Box 0 axis 1: min-max distance (0) below required minimum 1
12 boxes


Editing through `layer.bounding_boxes` is validated in the same way and recorded in the layer's undo history
(50 steps by default; `Ctrl+Z` / `Cmd+Z` in napari).

In [5]:
edited = layer.bounding_boxes
edited[0] += [0, 4, 4]  # move box 0 by 4 voxels in y and x
layer.bounding_boxes = edited
print("last step:", layer.get_undo_info()["undo_description"])

layer.undo()
print("after undo, box 0 is back:", np.array_equal(layer.bounding_boxes[0], boxes[0]))
layer.redo()
print("after redo, box 0 moved:  ", layer.bounding_boxes[0].tolist())

last step: Set boxes
after undo, box 0 is back: True
after redo, box 0 moved:   [[20.0, 37.0, 15.0], [29.0, 54.0, 32.0]]


## 2. A 2D view draws only the boxes on its slice

In a 2D view each box that intersects the current slice is drawn as one rectangle; the others are not drawn at all.
The layer finds them with a spatial index, so moving through the slices stays fast with many boxes.

In [6]:
for z in (5, 20, 35):
    viewer.dims.set_point(0, z)
    print(f"slice z = {z:2d}: {layer.nshapes:2d} of {len(layer.bounding_boxes)} boxes drawn")
viewer.close()

slice z =  5:  2 of 12 boxes drawn
slice z = 20:  3 of 12 boxes drawn
slice z = 35:  3 of 12 boxes drawn


## 3. Four synchronized viewers sharing one store

`create_synchronized_bbox_layers` creates one editable layer in the main viewer and read-only layers in the other viewers.
All of them display **one** `BBoxDataStore`: an edit is written once and every view updates from the store.
Each layer's update mode is taken from its viewer when the layers are created:
2D viewers update on every change (`"live"`), 3D viewers once at the end of an edit session such as a mouse drag (`"on_commit"`).
Set `ndisplay = 3` **before** creating the layers.

In [7]:
titles = ("XY (editable)", "XZ", "YZ", "3D")
viewers = [napari.Viewer(title=t, show=SHOW) for t in titles]
xy, xz, yz, v3d = viewers
for v in viewers:
    v.add_image(volume, name="volume")
xz.dims.order = (1, 0, 2)  # displays (z, x); slices along y
yz.dims.order = (2, 0, 1)  # displays (z, y); slices along x
v3d.dims.ndisplay = 3

layers = create_synchronized_bbox_layers(xy, [xz, yz, v3d], image_shape=volume.shape, edge_color="yellow")
main = layers[0]
main.add_boxes(boxes)
for title, lyr in zip(titles, layers):
    print(f"{title:14s} sync_mode={lyr.sync_mode:9s} editable={lyr.editable!s:5s} shapes drawn={lyr.nshapes}")

XY (editable)  sync_mode=live      editable=True  shapes drawn=2
XZ             sync_mode=live      editable=False shapes drawn=2
YZ             sync_mode=live      editable=False shapes drawn=2
3D             sync_mode=on_commit editable=False shapes drawn=12


### What a mouse drag does

A drag in the editable view opens an edit session on the store, writes the dragged box with `update_box` on every mouse move,
and closes the session on release. The same calls are available to your code, for example to stream boxes from an algorithm.
`store.update_box` is the low-level path: it does not validate the box and does not create an undo step
(use `main.bounding_boxes = ...` for that).

In [8]:
store = main.store  # the BBoxDataStore that all four layers display
i = 0
old = store.data[i].copy()  # store.data is a read-only view of the (N, 2, D) array
new = old + [0, 3, 3]

def box_in_3d_view():
    # a 3D layer draws one wireframe per box, in box order
    path = np.asarray(layers[3].data[i])
    return path.min(axis=0).round(1).tolist()

store.begin_edit_session()  # mouse press
store.update_box(i, new)  # mouse move (repeated during a drag)
print("during the drag, 3D view shows box min", box_in_3d_view(), "(old position)")
store.end_edit_session()  # mouse release
print("after release,   3D view shows box min", box_in_3d_view(), "(new position)")
store.update_box(i, old)  # put it back

during the drag, 3D view shows box min [20.0, 33.0, 11.0] (old position)
after release,   3D view shows box min [20.0, 36.0, 14.0] (new position)


## 4. Listening to changes and bulk edits

`store.register_listener` calls a function after every change. It receives a read-only view of all boxes and,
for single-box edits, the index of the changed box (`changed_idx`, else `None`).
Bound methods are held by weak reference; plain functions are kept alive until you unregister them.
Inside `store.batch_update()` all notifications are merged into one.

In [9]:
calls = []

def on_change(data, changed_idx=None):
    calls.append(changed_idx)

store.register_listener(on_change)
store.update_box(1, store.data[1])
print("single-box edit -> changed_idx", calls[-1])

calls.clear()
with store.batch_update():
    for k in range(5):
        store.update_box(k, store.data[k])
print("5 edits in batch_update ->", len(calls), "notification (changed_idx", calls[-1], ")")
store.unregister_listener(on_change)

Listener <function on_change at 0x147d23d80> cannot be weak-referenced; storing strong reference


single-box edit -> changed_idx 1
5 edits in batch_update -> 1 notification (changed_idx None )


## 5. Export and import

COCO and YOLO are 2D formats. By default every 3D box becomes one 2D box on each slice it covers
(`mode="per_slice"`); `mode="max_projection"` gives one box per object on a single image.
Widths and heights are `max - min`. The native format is the `(N, 2, D)` array itself (`.npy` or `.json`),
which the control panel's *Import BBoxes...* reads back.

In [10]:
out = Path(tempfile.mkdtemp(prefix="turbobox_"))
current = main.bounding_boxes

coco = boxes_to_coco(current, volume.shape, category_names={0: "blob"}, category_ids=[0] * len(current))
print(len(coco["images"]), "slice images,", len(coco["annotations"]), "annotations")
print(coco["images"][0], coco["annotations"][0], sep="\n")

yolo = boxes_to_yolo(current, volume.shape)  # {file name: file contents}
name, text = next(iter(yolo.items()))
print(len(yolo), "YOLO files; first line of", name + ":", text.splitlines()[0])

flat = boxes_to_coco(current, volume.shape, mode="max_projection")
print("max_projection:", len(flat["images"]), "image,", len(flat["annotations"]), "annotations")

(out / "coco.json").write_text(json.dumps(coco))
for fname, content in yolo.items():
    (out / fname).write_text(content)
np.save(out / "boxes.npy", current)  # native format

39 slice images, 120 annotations
{'id': 20, 'file_name': 'slice_0020.png', 'width': 128, 'height': 128}
{'id': 1, 'image_id': 20, 'category_id': 1, 'bbox': [11.0, 33.0, 17.0, 17.0], 'area': 289.0, 'iscrowd': 0}
39 YOLO files; first line of slice_0020.txt: 0 0.152344 0.324219 0.132812 0.132812
max_projection: 1 image, 12 annotations


In [11]:
loaded = np.load(out / "boxes.npy")
fresh = TurboBoxLayer(ndim=3, image_shape=volume.shape)
fresh.add_boxes(loaded)  # validated like any other write
print(len(fresh.bounding_boxes), "boxes imported; identical:", np.array_equal(fresh.bounding_boxes, current))

12 boxes imported; identical: True


## 6. Boxes as prompts

A box's `max` is inclusive, so a crop is `volume[min : max + 1]` on every axis.
Many promptable segmentation tools expect `(x, y, z)` order; reverse the last axis for them.

In [12]:
def crop(image, box):
    (z0, y0, x0), (z1, y1, x1) = np.asarray(box, dtype=int)
    return image[z0 : z1 + 1, y0 : y1 + 1, x0 : x1 + 1]

patch = crop(volume, current[0])
print("crop of box 0:", patch.shape, "max intensity", round(float(patch.max()), 2))

xyz = current[:, :, ::-1]  # [[x_min, y_min, z_min], [x_max, y_max, z_max]] per box
print("box 0 in (x, y, z) order:", xyz[0].tolist())

crop of box 0: (10, 18, 18) max intensity 0.99
box 0 in (x, y, z) order: [[11.0, 33.0, 20.0], [28.0, 50.0, 29.0]]


## 7. 2D images and the control panel

The same layer works for 2D images (`ndim=2`, boxes `[[y_min, x_min], [y_max, x_max]]`); exports then go to a single image.
The control panel (*Plugins > BBox Control Panel*) can also be added from code.

In [13]:
layer2d = TurboBoxLayer(ndim=2, image_shape=(256, 256))
layer2d.add_boxes([[[10, 20], [60, 90]], [[100, 100], [140, 200]]])
print(boxes_to_yolo(layer2d.bounding_boxes, (256, 256)))

dock, panel = xy.window.add_plugin_dock_widget("napari-turbobox", "BBox Control Panel")
print(type(panel).__name__, "-", panel.status_label.text())

{'image.txt': '0 0.214844 0.136719 0.273438 0.195312\n0 0.585938 0.468750 0.390625 0.156250\n'}


BoundingBoxControlWidget - 12 bounding box(es) (12 on this view). Click to select or use napari tools to edit.


In [14]:
if not SHOW:
    for v in viewers:
        v.close()